# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step template for loading and exploring the FAIR\textsuperscript{2} dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source

The dataset is described by a [Croissant schema](https://github.com/mlcommons/croissant) and is available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}:\n{metadata['description']}\n")
print(f"Identifier: {metadata['identifier']}")
print(f"License: {metadata['license']}")
print(f"Version: {metadata['version']}")


## 2. Data Overview

Review available record sets, fields, and their `@id` fields.

We'll list the record sets and fields defined in the schema. All references will use the `@id` for clarity and reproducibility.


In [ ]:
# Access the JSON-LD schema of the dataset
schema = dataset.metadata._jsonld

def extract_record_sets(schema):
    """
    Extract record set @id and names from the Croissant schema
    """
    record_sets = []
    for k, v in schema.items() if isinstance(schema, dict) else []:
        if v.get('@type') == 'cr:RecordSet':
            record_sets.append({'@id': v['@id'], 'name': v.get('schema:name', v.get('name', 'Unnamed Record Set'))})
    # Alternatively, loop top-level if recordSet listed on metadata
    if 'recordSet' in schema:
        rs_list = schema['recordSet'] if isinstance(schema['recordSet'], list) else [schema['recordSet']]
        for rs in rs_list:
            if isinstance(rs, dict):
                record_sets.append({'@id': rs['@id'], 'name': rs.get('name', 'Unnamed Record Set')})
            elif isinstance(rs, str):
                # The actual node will be elsewhere
                pass
    # Fallback: look through all values
    results = []
    objects = schema.values() if isinstance(schema, dict) else []
    for obj in objects:
        if isinstance(obj, dict) and obj.get('@type') in ('cr:RecordSet', 'RecordSet'):
            results.append({'@id': obj['@id'], 'name': obj.get('name', obj.get('schema:name', 'Unnamed Record Set'))})
    if len(results) > 0:
        return results
    if len(record_sets) > 0:
        return record_sets
    # If only one RecordSet, sometimes everything is flat at top level
    if ('@type' in schema and schema['@type'] in ('cr:RecordSet', 'RecordSet')):
        return [{ '@id': schema['@id'], 'name': schema.get('name', schema.get('schema:name', 'Unnamed Record Set')) }]
    return []

# Find all cr:RecordSet entities by @id
from collections.abc import Mapping

def find_record_sets(obj):
    record_sets = []
    if isinstance(obj, Mapping):
        if obj.get('@type') in ['cr:RecordSet', 'RecordSet']:
            record_sets.append(obj)
        for v in obj.values():
            record_sets.extend(find_record_sets(v))
    elif isinstance(obj, list):
        for v in obj:
            record_sets.extend(find_record_sets(v))
    return record_sets

record_sets = find_record_sets(schema)
print(f"Found {len(record_sets)} record sets.\n")
record_set_ids = []
for record_set in record_sets:
    print(f"- RecordSet Name: {record_set.get('name', record_set.get('schema:name', 'Unnamed'))}")
    print(f"  @id: {record_set['@id']}")
    record_set_ids.append(record_set['@id'])
    # Print fields for this record set
    fields = record_set.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        if isinstance(field, dict):
            print(f"    - {field.get('name', field.get('schema:name', 'Unnamed Field'))} (@id: {field['@id']})")
        elif isinstance(field, str):
            print(f"    - @id: {field}")
    print()

# For next section, pick first record set for extraction
if len(record_set_ids) > 0:
    print(f"Example record set for extraction: {record_set_ids[0]}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. We use the `@id` of the record set and will display the corresponding fields (`@id` for each column).

We'll iterate through all record sets found above and load them into pandas DataFrames for further analysis.


In [ ]:
# Extract data from each record set by @id
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))  # Each record is a dict mapping field @id to value
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set @id: {record_set_id}")
        print(f"Fields (column @id): {list(df.columns)}\n")
    else:
        print(f"No records loaded for record set @id: {record_set_id}\n")

# Use the first found record set as primary for exploration
primary_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if primary_record_set_id and primary_record_set_id in dataframes:
    print("First 5 rows of primary record set:")
    display(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on a specific field, normalizing numeric columns, and grouping data. All columns are referenced by their `@id`.

Examples will identify a numeric column for filtering and demonstrate normalization and grouping. Change the field `@id`s as needed based on DataFrame columns from previous section.

In [ ]:
import numpy as np

# Choose a numeric field for demonstration. (Replace these with actual field @id from your schema)
# For example, suppose the field @id for "Age at second CRC" is 'https://api.app.sen.science/frontiers/7862866/field/age_at_second_crc'
# Let's list all the fields/columns found:
if primary_record_set_id:
    df = dataframes[primary_record_set_id]
    print("Available columns (@id):")
    print(df.columns.tolist())

# ---- Example workflow (replace with actual field @id that is numeric) ----
# Suppose we pick the first numeric column found
# We'll look for 'age' or 'interval' or similar field in the column names for this demonstration.
numeric_field = None
for col in df.columns:
    if any(word in col.lower() for word in ['age', 'interval', 'year', 'number']):
        if np.issubdtype(df[col].dropna().apply(type).mode()[0], (int, float)) or pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    # Or try to infer numeric type
if numeric_field is None:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

if numeric_field:
    print(f"Chosen numeric field '@id': {numeric_field}")
    # Set filter threshold for demonstration
    threshold = 60  # e.g., filter 'age' > 60
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (rows: {len(filtered_df)}):")
    display(filtered_df.head())

    # Normalization
    field_norm = f"{numeric_field}_normalized"
    filtered_df[field_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, field_norm]].head())

    # Group by another field (e.g., 'sex' or 'diagnosis')
    group_field = None
    for col in df.columns:
        if any(k in col.lower() for k in ['sex', 'gender', 'group', 'anatomical', 'location']):
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nMean {numeric_field} grouped by {group_field}:")
        display(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No numeric field identified for demonstration. Please check dataset columns above.")

## 5. Visualization

Visualize distributions or relationships between fields. All visual references are by `@id`.

We'll use matplotlib and seaborn for demonstration. (Feel free to adjust plot style or fields as appropriate.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the chosen numeric field
if primary_record_set_id and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field} (@id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If a group field was found, show group differences
    if group_field:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field} (@id)")
        plt.xticks(rotation=30)
        plt.show()


## 6. Conclusion

In this notebook, we loaded and explored the FAIR\textsuperscript{2} dataset describing the clinicopathological and molecular characteristics of second primary colorectal cancer in cancer survivors. Using the `mlcroissant` library, we referenced all dataset entities by their unique `@id` fields, extracted records, explored numeric and categorical fields, performed filtering, normalization, grouping, and produced visualizations.

Key observations:
- The dataset is richly described by Croissant metadata, enabling reproducible analytics.
- Numeric and categorical data can be referenced and manipulated by `@id` consistently.
- Data visualization highlights important patterns (e.g., age distribution, anatomical location, etc.).

Further analysis can be performed by referencing additional `@id` fields for specific research questions or applying advanced statistical analysis.